In [5]:

# ============================================================
# MODULE 1 - CINESTREAM DATA FOUNDATION
# File suggestion: notebooks/m1_explore.ipynb
# ============================================================

import pandas as pd
import numpy as np

# ------------------------------------------------------------
# 1) LOAD DATA
# ------------------------------------------------------------


df = pd.read_csv(r"C:\Users\muhju\Downloads\4.2\CineStream_Catalog - CineStream_Catalog.csv")

print("Dataset loaded successfully.\n")


# ------------------------------------------------------------
# 2) BASIC EXPLORATION
# ------------------------------------------------------------
print("========== BASIC DATA EXPLORATION ==========")

# Shape
print("\n1. Shape of dataset:")
print(df.shape)

# Column names
print("\n2. Column names:")
print(df.columns.tolist())

# Data types and non-null counts
print("\n3. DataFrame info:")
print(df.info())

# First few rows
print("\n4. First 5 rows:")
print(df.head())

# Missing values
print("\n5. Missing values in each column:")
print(df.isnull().sum())

# Duplicate rows
print("\n6. Number of duplicate rows:")
print(df.duplicated().sum())


# ------------------------------------------------------------
# 3) UNIQUE VALUES OF CATEGORICAL COLUMNS
# ------------------------------------------------------------
print("\n========== UNIQUE VALUES OF CATEGORICAL COLUMNS ==========")

categorical_cols = [
    "Type", "Genre", "Language", "Country",
    "Director", "AgeRating", "TrendingStatus"
]

for col in categorical_cols:
    print(f"\n--- {col} ---")
    print(df[col].dropna().unique())
    print(f"Unique count in {col}: {df[col].nunique(dropna=True)}")


# ------------------------------------------------------------
# 4) DATA QUALITY REPORT
# ------------------------------------------------------------
print("\n========== DATA QUALITY REPORT ==========")

# Keep a copy for checking issues before cleaning
raw_df = df.copy()

# 4.1 Text columns
text_cols = raw_df.select_dtypes(include="object").columns.tolist()

# 4.2 Trailing / leading spaces count
print("\n1. Trailing / leading spaces check:")
for col in text_cols:
    # convert to string only for checking, keep NaN safe
    count_spaces = raw_df[col].dropna().astype(str).str.contains(r"^\s|\s$", regex=True).sum()
    if count_spaces > 0:
        print(f"{col}: {count_spaces} rows with leading/trailing spaces")

# 4.3 Case mismatch in Type
print("\n2. Case mismatch check in Type:")
print("Unique values before cleaning:", raw_df["Type"].dropna().unique())

# 4.4 Blank Director and AddedDate
print("\n3. Blank / missing values:")
print("Missing Director:", raw_df["Director"].isnull().sum())
print("Missing AddedDate:", raw_df["AddedDate"].isnull().sum())

# 4.5 Out-of-range IMDbScore and CriticRating
imdb_invalid = raw_df[(raw_df["IMDbScore"] < 1) | (raw_df["IMDbScore"] > 10)]
critic_invalid = raw_df[(raw_df["CriticRating"] < 1) | (raw_df["CriticRating"] > 10)]

print("\n4. Out-of-range ratings:")
print("Out-of-range IMDbScore rows:", len(imdb_invalid))
if len(imdb_invalid) > 0:
    print(imdb_invalid[["ContentID", "Title", "IMDbScore"]])

print("Out-of-range CriticRating rows:", len(critic_invalid))
if len(critic_invalid) > 0:
    print(critic_invalid[["ContentID", "Title", "CriticRating"]])

# 4.6 Absurd runtime
# PRD says one absurd runtime exists. We'll flag very large runtime values.
runtime_outliers = raw_df[raw_df["RuntimeMinutes"] > 1000]

print("\n5. Absurd RuntimeMinutes:")
print("Rows with RuntimeMinutes > 1000:", len(runtime_outliers))
if len(runtime_outliers) > 0:
    print(runtime_outliers[["ContentID", "Title", "RuntimeMinutes"]])

# 4.7 Negative SubscribersGainedThousands
negative_subs = raw_df[raw_df["SubscribersGainedThousands"] < 0]

print("\n6. Negative SubscribersGainedThousands:")
print("Rows with negative subscriber counts:", len(negative_subs))
if len(negative_subs) > 0:
    print(negative_subs[["ContentID", "Title", "SubscribersGainedThousands"]].head(10))

# 4.8 Duplicate rows
duplicate_rows = raw_df[raw_df.duplicated()]
print("\n7. Duplicate rows:")
print("Number of duplicate rows:", duplicate_rows.shape[0])
if duplicate_rows.shape[0] > 0:
    print(duplicate_rows)


# ------------------------------------------------------------
# 5) CLEAN THE DATA
# ------------------------------------------------------------
print("\n========== CLEANING DATA ==========")

clean_df = df.copy()

# 5.1 Strip whitespace in all text columns
for col in text_cols:
    clean_df[col] = clean_df[col].astype("string").str.strip()

# 5.2 Standardise case in Type
# This converts movie -> Movie, series -> Series, documentary -> Documentary
clean_df["Type"] = clean_df["Type"].str.title()

# 5.3 Drop duplicate rows
before_rows = clean_df.shape[0]
clean_df = clean_df.drop_duplicates()
after_rows = clean_df.shape[0]
print(f"Duplicates removed: {before_rows - after_rows}")

# 5.4 Replace out-of-range IMDbScore with NaN
clean_df.loc[
    (clean_df["IMDbScore"] < 1) | (clean_df["IMDbScore"] > 10),
    "IMDbScore"
] = np.nan

# 5.5 Replace out-of-range CriticRating with NaN
clean_df.loc[
    (clean_df["CriticRating"] < 1) | (clean_df["CriticRating"] > 10),
    "CriticRating"
] = np.nan

# 5.6 Replace absurd RuntimeMinutes with NaN
# Based on PRD, there is an absurd runtime. We treat very large values as bad data.
clean_df.loc[clean_df["RuntimeMinutes"] > 1000, "RuntimeMinutes"] = np.nan

# 5.7 Replace negative subscriber counts with 0
clean_df.loc[
    clean_df["SubscribersGainedThousands"] < 0,
    "SubscribersGainedThousands"
] = 0

# 5.8 Convert AddedDate to datetime
clean_df["AddedDate"] = pd.to_datetime(clean_df["AddedDate"], errors="coerce")

# Optional: convert Director blank strings to NaN if any blank strings exist
clean_df["Director"] = clean_df["Director"].replace("", np.nan)

print("Cleaning completed.\n")


# ------------------------------------------------------------
# 6) CREATE DERIVED COLUMNS
# ------------------------------------------------------------
print("========== CREATING DERIVED COLUMNS ==========")

# Profit_Cr = RevenueCr - ProductionCostCr
clean_df["Profit_Cr"] = clean_df["RevenueCr"] - clean_df["ProductionCostCr"]

# ROI_Pct = Profit_Cr / ProductionCostCr * 100
clean_df["ROI_Pct"] = np.where(
    clean_df["ProductionCostCr"] > 0,
    (clean_df["Profit_Cr"] / clean_df["ProductionCostCr"]) * 100,
    np.nan
)

# Performance_Band
clean_df["Performance_Band"] = np.where(
    clean_df["Profit_Cr"] > 20,
    "Hit",
    np.where(
        clean_df["Profit_Cr"] < 0,
        "Flop",
        "Break-even"
    )
)

print(clean_df[["Title", "ProductionCostCr", "RevenueCr", "Profit_Cr", "ROI_Pct", "Performance_Band"]].head())


# ------------------------------------------------------------
# 7) POST-CLEANING CHECK
# ------------------------------------------------------------
print("\n========== POST-CLEANING CHECK ==========")
print("Final shape:", clean_df.shape)
print("\nMissing values after cleaning:")
print(clean_df.isnull().sum())

print("\nUnique values in Type after cleaning:")
print(clean_df["Type"].unique())

print("\nSample cleaned data:")
print(clean_df.head())


# ------------------------------------------------------------
# 8) SAVE CLEANED DATASET
# ------------------------------------------------------------
output_file = "cleaned_cinestream.csv"
clean_df.to_csv(output_file, index=False)

print(f"\nCleaned dataset saved successfully as: {output_file}")




Dataset loaded successfully.

========== BASIC DATA EXPLORATION ==========

1. Shape of dataset:
(524, 23)

2. Column names:
['ContentID', 'Title', 'Type', 'Genre', 'Language', 'Country', 'Director', 'CastCount', 'ReleaseYear', 'AddedDate', 'RuntimeMinutes', 'EpisodeCount', 'AgeRating', 'IMDbScore', 'ViewsMillions', 'WatchHoursMillions', 'AvgCompletionPct', 'AwardsWon', 'ProductionCostCr', 'RevenueCr', 'SubscribersGainedThousands', 'TrendingStatus', 'CriticRating']

3. DataFrame info:
<class 'pandas.DataFrame'>
RangeIndex: 524 entries, 0 to 523
Data columns (total 23 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   ContentID                   524 non-null    str    
 1   Title                       524 non-null    str    
 2   Type                        524 non-null    str    
 3   Genre                       524 non-null    str    
 4   Language                    524 non-null    str    
 5   Country     

C:\Users\muhju\AppData\Local\Temp\ipykernel_22668\1494057561.py:74: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  text_cols = raw_df.select_dtypes(include="object").columns.tolist()
